# 8. WATERec: aspect-preserving OCR

WATERec is a separate model family, not a PARSeq preset. It keeps two-dimensional image patches and uses a small autoregressive character decoder. Use this notebook for its complete training, evaluation and TensorRT workflow; notebooks 1–7 remain the PARSeq workflow.

| Option | Use | Generic initialization | Patch budget |
| --- | --- | --- | --- |
| `waterec_alpr.yaml` | Plates, including varied layouts | RS | 512 |
| `waterec_price.yaml` | Price tags, spaces and punctuation | R | 512 |

The patch budget preserves aspect ratio; it is not a fixed square image resolution. Different budgets need separate accuracy checks and exported engines. The validated TensorRT deployment is single-crop batch 1.

Our trained plate TensorRT model scored **2475/2693 (91.90%)** on validation and **880/984 (89.43%)** on reviewed unseen plates. The optimized Florence reference scored 2381 and 857 respectively. On an RTX5090, the default cached decoder measured about **2.64 ms** including file decoding, or **1.68 ms** from decoded RGB, at **688 MiB** driver-accounted GPU memory. These are hardware/dataset-specific measurements, not a 1 ms guarantee.

Price WATERec is an experimental alternative: after full training it scored 129/144 on clear AI-reviewed numeric prices, versus Florence 133/144 and the separate compact PP-OCR price model 131/144. Do not assume its plate advantage transfers to price tags. AI-reviewed labels are not human-certified.

Vocabulary: printable ASCII, with an explicit space token for price tags; up to 25 characters. Non-ASCII scripts/currency require extending the head and retraining. These benchmark scores do not establish universal multilingual OCR.


In [ ]:
from pathlib import Path
import os, sys, subprocess
ROOT = Path.cwd().resolve()
if not (ROOT / "parseq_ocr").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "parseq_ocr").is_dir():
    raise RuntimeError("Open from the repository or notebooks directory")
sys.path.insert(0, str(ROOT))
from parseq_ocr.waterec_config import load_config
CONFIG_PATH = os.environ.get("WATEREC_CONFIG", "configs/waterec_alpr.yaml")
config = load_config(CONFIG_PATH)
print(CONFIG_PATH, config["variant"], config["tokens"], config["output_dir"])
def run(*args):
    return subprocess.run([sys.executable, "-m", "parseq_ocr.waterec_cli", *args,
                           "--config", CONFIG_PATH], cwd=ROOT, check=True)


## Prepare and train

Place cropped images and matching UTF-8 `.txt` transcriptions under the configured `dataset_root/train` and `dataset_root/val`. Use complete text, including decimal punctuation. Training selects epochs only on an internal training holdout, then refits from generic initialization. Evaluation images must remain outside that selection. The initial epochs train the decoder only; subsequent epochs adapt the encoder as well.

The source release contains no private dataset or trained checkpoint. Reproduction depends on supplying the same data and recipe; training a new dataset will not reproduce the published counts automatically.


In [ ]:
RUN_TRAINING = False  # Enable after preparing and checking your training data.
if RUN_TRAINING:
    run("train")


## Inspect native predictions and evaluate

Evaluate exact text as well as the actual numeric value for price tags. Removing punctuation can hide decimal errors, so alphanumeric agreement alone is insufficient. Visually review missing or ambiguous annotations before interpreting small accuracy differences.


In [ ]:
IMAGE_PATH = ROOT / "dataset/val/example.jpg"
RUN_NATIVE = False
if RUN_NATIVE:
    run("infer", "--image", str(IMAGE_PATH))
    run("evaluate", "--folder", str(config["dataset_root"] / "val"),
        "--output", str(config["output_dir"] / "evaluation_native"))


## Export and build TensorRT

Export produces an encoder/cross-attention graph and a cached decoder-step graph. Build engines on the target NVIDIA GPU with the documented TensorRT dependencies. The runtime preserves explicit post-RoPE attention barriers and FP32 normalization/softmax to avoid a measured attention-fusion error. It uses a single-crop engine; a Python list is processed sequentially, not as a batched GPU execution.

FP16 conversion can change predictions: the validated plate engine differed from native FP32 on two of 3677 images, with one fewer correct validation result and one more correct unseen result. Always evaluate your own engines.


In [ ]:
RUN_EXPORT = False
RUN_BUILD = False
if RUN_EXPORT:
    run("export")
if RUN_BUILD:
    run("build")


In [ ]:
RUN_TENSORRT = False
if RUN_TENSORRT:
    run("infer", "--backend", "tensorrt", "--image", str(IMAGE_PATH))
    run("evaluate", "--backend", "tensorrt", "--folder", str(config["dataset_root"] / "val"),
        "--output", str(config["output_dir"] / "evaluation_tensorrt"))


## Compare latency on the same images

Use an idle GPU and fresh processes. Report file-to-text and predecoded RGB-to-text separately; detection, network transfer and model loading are excluded. The commands below use the same folder and five warmups. Memory includes the CUDA context, not just weights. Avoid comparing one model's GPU-only timing against another model's complete preprocessing and inference.

Rotation correction is not enabled by this workflow. Experimental orientation classification greatly improved quarter-turn recognition but slightly harmed upright accuracy; it must be validated as an optional pipeline rather than assumed lossless.


In [ ]:
RUN_BENCHMARK = False
if RUN_BENCHMARK:
    for predecoded in (False, True):
        scope = "predecoded" if predecoded else "file_to_text"
        extra = ["--predecoded"] if predecoded else []
        run("benchmark", "--backend", "tensorrt", "--folder", str(config["dataset_root"] / "val"),
            "--output", str(config["output_dir"] / f"benchmark_{scope}.json"), *extra)


## Optional quarter-turn correction

This is a separate, explicitly enabled pipeline. Train its small orientation head against the **exact finalized OCR encoder**, using only training images and a disjoint training holdout. Each image generates known 0°, 90°, 180° and 270° rotations. A final OCR encoder may already have learned from those holdout images during text training; that inherited exposure must be disclosed. External evaluation images never belong in these manifests.

Supply two JSON lists of records with `path` fields. Relative image paths resolve against the manifest's directory. Export checks the encoder checkpoint hash so that a head cannot silently be reused with a different encoder. Upright crops reuse the first encoder output; corrected crops require another encoder pass. The tested head slightly harmed upright accuracy, so enabling it is a task-dependent choice.

The experimental conditional-decoder pipeline scored 706/788, 706/788 and 707/788 on 90°, 180° and 270° rotations. On upright images it scored 879/984 versus 880/984 without correction. Its predecoded timing used a different decoder variant from this public default; benchmark the public pipeline directly rather than copying those timings.


In [ ]:
RUN_ORIENTATION_TRAINING = False
RUN_ORIENTATION_BUILD = False
RUN_ORIENTATION_INFERENCE = False
FIT_MANIFEST = ROOT / "dataset/orientation_fit.json"
HELD_MANIFEST = ROOT / "dataset/orientation_held.json"
if RUN_ORIENTATION_TRAINING:
    run("train-orientation", "--fit-manifest", str(FIT_MANIFEST),
        "--held-manifest", str(HELD_MANIFEST))
if RUN_ORIENTATION_BUILD:
    run("export-orientation")
    run("build-orientation")
if RUN_ORIENTATION_INFERENCE:
    run("infer", "--backend", "tensorrt", "--orientation", "--image", str(IMAGE_PATH))
    run("evaluate", "--backend", "tensorrt", "--orientation",
        "--folder", str(config["dataset_root"] / "val"),
        "--output", str(config["output_dir"] / "evaluation_orientation"))


## Optional exact CUDA preprocessing

The default remains CPU Pillow. With existing 512-patch TensorRT engines, `--gpu-preprocess` moves exact bicubic resizing, normalization and patch packing to CUDA. JPEG/image-file decoding and RGB conversion remain on the CPU; this is not GPU JPEG decoding. It also works with the explicitly enabled orientation pipeline. No model retraining or re-export is required.

This opt-in path requires Linux, a C compiler (`CC` or `cc`) and the CUDA toolkit's `nvcc`. The supplied Docker image installs `build-essential`. The coefficient library is built from source into `${XDG_CACHE_HOME:-~/.cache}/parseq-ocr/resize`; CUDA kernels compile through PyCUDA. Compilation and initial setup must not be counted as steady-state inference. The source contains `.c`, `.cu` and the Pillow license, with no compiled binaries.

Public checks preserved all 4,120 prepared inputs/predictions and 4,136 orientation predictions/angles, plus 80 queued cache/buffer lifecycle cases. CPU preprocessing remains available by omitting the flag. Benchmark the new public option on your target GPU: earlier conditional-decoder and CPU-preprocessing timings do not measure this new option. Report encoded and predecoded scopes separately; benchmark JSON records preprocessing/source identity.


| Pipeline | Preprocessing | File-to-text mean ms | Predecoded RGB-to-text mean ms | GPU MiB |
| --- | --- | ---: | ---: | ---: |
| Default OCR | CPU Pillow | 2.709 | 1.643 | 688 |
| Default OCR | GPU opt-in | 2.273 | 1.261 | 694 |
| Optional orientation | CPU Pillow | 2.825 | 1.805 | 688 |
| Optional orientation | GPU opt-in | 2.461 | 1.411 | 694 |

Fresh isolated processes on RTX 5090, the same 256 mixed plate crops, batch 1 and five warmups; the public one-step cached decoder is unchanged. Both scopes include resize, normalization, patch packing, inference and text decode. File-to-text additionally includes local image-file read/JPEG decode; network, detection and model loading are excluded. These are measured values, not universal guarantees. Orientation timing depends on the frequency of corrections; rotated crops may need a second encoder pass.

In [ ]:
RUN_GPU_PREPROCESSING = False
if RUN_GPU_PREPROCESSING:
    run("infer", "--backend", "tensorrt", "--gpu-preprocess", "--image", str(IMAGE_PATH))
    run("evaluate", "--backend", "tensorrt", "--gpu-preprocess",
        "--folder", str(config["dataset_root"] / "val"),
        "--output", str(config["output_dir"] / "evaluation_gpu_preprocess"))
    for predecoded in (False, True):
        scope = "rgb" if predecoded else "file_to_text"
        extra = ["--predecoded"] if predecoded else []
        run("benchmark", "--backend", "tensorrt", "--gpu-preprocess",
            "--folder", str(config["dataset_root"] / "val"),
            "--output", str(config["output_dir"] / f"benchmark_gpu_{scope}.json"), *extra)
    # Add --orientation only after training/exporting/building its separate head.
